In [ ]:
# !pip install -q "xformers<0.0.27" "trl<0.9.0" peft bitsandbytes "transformers>=4.43.2" accelerate hf_transfer unsloth[colab]
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
# pip install unsloth_zoo

In [ ]:
# !pip install -q datasets

In [ ]:
from unsloth import FastLanguageModel

import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback, DataCollatorWithPadding, EarlyStoppingCallback
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
torch.cuda.is_available()

True

In [ ]:
DATASET_TRAIN = 'pvs-blocks-single-new-train.csv'
DATASET_VAL = 'pvs-blocks-single-new-val.csv'

EXP = '2025.04.11-1-pvs-masked-clean-data'

In [ ]:
CHECKPOINT = 'unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit'

In [ ]:
# !huggingface-cli download 'unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit'

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    CHECKPOINT,
    max_seq_length=512,
    dtype=torch.bfloat16,
    load_in_4bit = True,
    device_map='auto'
)

==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.064 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), 

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    max_seq_length=512,
    lora_dropout=0.00,
)

Unsloth 2025.3.19 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


In [ ]:
df = pd.read_csv(DATASET_TRAIN)
df

,samples,sequent,command,parameters
0,"{-1} occurrences(l2!1)(nth(l1!1, i!1)) = 0\n[...","{-1} occurrences(l2!1)(nth(l1!1, i!1)) = 0\n[...",LEMMA,"""member_implies_pos_occurrences"""
1,[-1] reflexive?(leq)\n{-2} (leq(x!1`seq(x!2)...,[-1] reflexive?(leq)\n{-2} (leq(x!1`seq(x!2)...,ASSERT,NaN
2,[-1] x!2 < 1 + length(x!1)\n[-2] i < length(...,[-1] x!2 < 1 + length(x!1)\n[-2] i < length(...,PROP,NaN
3,[-1] i > j\n[-2] n > 0\n{-3} (FORALL (i: be...,[-1] i > j\n[-2] n > 0\n{-3} (FORALL (i: be...,FLATTEN,NaN
4,"{-1} FORALL (l1: list[T], x: T): length(g_ele...","{-1} FORALL (l1: list[T], x: T): length(g_ele...",GRIND,NaN
...,...,...,...,...
9727,"[-1] preorder?(leq)\n{-2} FORALL (x: T), (y:...","[-1] preorder?(leq)\n{-2} FORALL (x: T), (y:...",INST,?
9728,{-1} i < length(smallerones(x!1`seqfj`seq(1))...,{-1} i < length(smallerones(x!1`seqfj`seq(1))...,ASSERT,NaN
9729,"[-1] gt(x!1`seq(ind_lc(x!2)), x!1`seq(x!2))\n...","[-1] gt(x!1`seq(ind_lc(x!2)), x!1`seq(x!2))\n...",GRIND,NaN
9730,|-------\n{1} FORALL (l: list[T]): null?(l...,|-------\n{1} FORALL (l: list[T]): null?(l...,GRIND,NaN


In [ ]:
ds = DatasetDict({
    "train": Dataset.from_pandas(df),
    "val": Dataset.from_pandas(pd.read_csv(DATASET_VAL))
})
ds

DatasetDict({
    train: Dataset({
        features: ['samples', 'sequent', 'command', 'parameters'],
        num_rows: 9732
    })
    val: Dataset({
        features: ['samples', 'sequent', 'command', 'parameters'],
        num_rows: 2794
    })
})

In [ ]:
import re

from IPython.display import HTML


MASK_PATTERNS = [
    (r'(?<=Rule\?) \(.*?\)\n', re.DOTALL)
]

SPACE = 'Ġ'
LINE_BREAK = 'Ċ'


def invert_labels(ids, labels):
    return [(id if label < 0 else -100 ) for id, label in zip(ids, labels)]



def mask_subsequence(ids, subtext_ids):
    i = 0
    while i <= len(ids) - len(subtext_ids):
        if ids[i:i + len(subtext_ids)] == subtext_ids:
            ids[i:i + len(subtext_ids)] = [-100] * len(subtext_ids)
            i += len(subtext_ids)
        else:
            i += 1

    return ids


def mask_everything_except(text="", ids=None):
    if ids is None:
        ids = tokenizer(text, add_special_tokens=False).input_ids

    labels = ids.copy()
    for pattern, flags in MASK_PATTERNS:
        for match in re.findall(pattern, text, flags=flags):
            subtext_ids = tokenizer(match, add_special_tokens=False).input_ids
            # print(match, subtext_ids)
            labels = mask_subsequence(labels, subtext_ids)

    return ids, invert_labels(ids, labels)


def mark_output(ids, labels, html=True):
    tokens = tokenizer.convert_ids_to_tokens(ids)
    text, not_masking = '', False
    for token, label in zip(tokens, labels):
        if not not_masking and label >= 0:
            text += '<mark>'
            not_masking = True
        elif not_masking and label < 0:
            text += '</mark>'
            not_masking = False

        text += token

    if not_masking:
        text += '</mark>'

    text = re.sub(r'<mark>( +)', r'\1<mark>', text)
    text = text.replace(SPACE, ' ').replace(LINE_BREAK, '<br>' if html else '\n')
    return f'<pre>{text}</pre>'



tokens, labels = mask_everything_except(ds["train"][0]["samples"])
# print(len(labels), labels)
marked_html = mark_output(tokens, labels)
# print(marked_html)
HTML(marked_html)

In [ ]:
def tokenize(sample):
    tokenized = tokenizer(sample["samples"], truncation=True, max_length=512, return_attention_mask=True)
    tokenized["labels"] = mask_everything_except(sample["samples"], ids=tokenized.input_ids)[1]

    return tokenized


tokenized_ds = ds.map(tokenize, num_proc=4, batched=False).remove_columns(ds["train"].features)

Map (num_proc=4):   0%|          | 0/9732 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2794 [00:00<?, ? examples/s]

In [ ]:
tokenized_ds["train"][1]["labels"][-20:]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 320,
 24745,
 340]

### tokenized_ds

In [ ]:
tokenizer.pad_token_id

128004

In [ ]:
class CLMDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        labels = [sample.pop("labels") for sample in features]

        batch = super().__call__(features)

        batch['labels'] = batch['input_ids'].clone()
        for i, labels_ in enumerate(labels):
            labels_tensor = torch.LongTensor(labels_)
            if tokenizer.padding_side == 'right':
                batch['labels'][i, :len(labels_)] = labels_tensor
            else:
                batch['labels'][i, -len(labels_):] = labels_tensor


        if self.tokenizer.pad_token_id is not None:
                batch['labels'][batch['labels'] == self.tokenizer.pad_token_id] = -100

        return batch


data_colator = CLMDataCollator(tokenizer)

In [ ]:
callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
batch_size = 256
logs_steps = len(ds["train"]) // batch_size


args = TrainingArguments(
    output_dir='./results2',
    num_train_epochs = 20,
    per_device_train_batch_size = batch_size,  # 128,
    per_device_eval_batch_size = 128,
    # optim = 'paged_adamw_8bit',
    optim='adamw_torch',
    # gradient_checkpointing=True,
    bf16=True,
    # bf16=True,
    logging_steps=logs_steps,
    report_to = "none",
    eval_strategy="steps",
    eval_steps=logs_steps,
    save_steps= 2 * logs_steps,
    push_to_hub=False,
    gradient_accumulation_steps=1,
    # warmup_ratio=.03,
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model,
    args,
    data_collator=data_colator,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["val"],
    callbacks=callbacks
)

In [ ]:
info = trainer.train()
info

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,732 | Num Epochs = 20 | Total steps = 780
O^O/ \_/ \    Batch size per device = 256 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (256 x 1 x 1) = 256
 "-____-"     Trainable parameters = 5,636,096/1,000,000,000 (0.56% trained)


Step,Training Loss,Validation Loss
38,0.879000,0.784682
76,0.696400,0.680282
114,0.585600,0.613905
152,0.545300,0.568228
190,0.477500,0.531854
228,0.434500,0.506674
266,0.388700,0.497994
304,0.352900,0.485156
342,0.341200,0.479821
380,0.297400,0.486899


TrainOutput(global_step=418, training_loss=0.4794149398803711, metrics={'train_runtime': 10801.7937, 'train_samples_per_second': 18.019, 'train_steps_per_second': 0.072, 'total_flos': 3.1414605367187866e+17, 'train_loss': 0.4794149398803711, 'epoch': 10.717948717948717})

In [ ]:
info

TrainOutput(global_step=418, training_loss=0.4794149398803711, metrics={'train_runtime': 10801.7937, 'train_samples_per_second': 18.019, 'train_steps_per_second': 0.072, 'total_flos': 3.1414605367187866e+17, 'train_loss': 0.4794149398803711, 'epoch': 10.717948717948717})

In [ ]:
model.save_pretrained(f"{EXP}-lora")
tokenizer.save_pretrained(f"{EXP}-lora")

('2025.04.11-1-pvs-masked-clean-data-lora/tokenizer_config.json',
 '2025.04.11-1-pvs-masked-clean-data-lora/special_tokens_map.json',
 '2025.04.11-1-pvs-masked-clean-data-lora/tokenizer.json')

In [ ]:
model.save_pretrained_gguf(f"{EXP}-lora", tokenizer, quantization_method = "q8_0")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 2.97 out of 31.35 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|█████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 53.44it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at 2025.04.11-1-pvs-masked-clean-data-lora into q8_0 GGUF format.
The output location will be /home/niksonber/experiments/2025.04.11-1-pvs-masked-clean-data-lora/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: 2025.04.11-1-pvs-masked-clean-data-lora
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:gguf: loading model part 'model.safetensors'
INFO:hf-to-gguf:token_embd.weight,      

In [ ]:
import os

os.environ ["EXP"] = EXP

In [ ]:
!rm 2025.04.11-1-pvs-masked-clean-data-lora/model.safetensors

In [ ]:
!tar -cvf "$EXP-lora.tar.gz" "$EXP-lora"

2025.04.11-1-pvs-masked-clean-data-lora/
2025.04.11-1-pvs-masked-clean-data-lora/special_tokens_map.json
2025.04.11-1-pvs-masked-clean-data-lora/README.md
2025.04.11-1-pvs-masked-clean-data-lora/tokenizer_config.json
2025.04.11-1-pvs-masked-clean-data-lora/config.json
2025.04.11-1-pvs-masked-clean-data-lora/adapter_config.json
2025.04.11-1-pvs-masked-clean-data-lora/adapter_model.safetensors
2025.04.11-1-pvs-masked-clean-data-lora/unsloth.Q8_0.gguf
2025.04.11-1-pvs-masked-clean-data-lora/generation_config.json
2025.04.11-1-pvs-masked-clean-data-lora/tokenizer.json


In [ ]:
# !python /home/niksonber/experiments/llama.cpp/convert_lora_to_gguf.py "$EXP-lora"

INFO:lora-to-gguf:Loading base model from Hugging Face: unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:lora-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_a, torch.float32 --> F32, shape = {8192, 8}
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_b, torch.float32 --> F32, shape = {8, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_a, torch.float32 --> F32, shape = {2048, 8}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_b, torch.float32 --> F32, shape = {8, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_a,  torch.float32 --> F32, shape = {2048, 8}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_b,  torch.float32 --> F32, shape = {8, 8192}
INFO:hf-to-gguf:blk.0.attn_k.weight.lora_a,  torch.float32 --> F32, shape = {2048, 8}
INFO:hf-to-gguf:blk.0.attn_k.weight.lora_b,  torch.float32 --> F32, shape = {8, 512}
INFO:hf-to-gguf:blk.0.attn_output.weight.lora_a, torch.float32 --> F32, shape = {2048, 8}
INFO:hf-to

# Inference

In [ ]:
!mv "$EXP-lora/config.json" "$EXP-lora/config2.json"

In [ ]:
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name = f"{EXP}-lora",
    max_seq_length = 512,
    dtype=torch.bfloat16,
    load_in_4bit = True,
    local_files_only=True,
    device_map="auto"
)


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.064 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
FastLanguageModel.for_inference(model2)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

In [ ]:
SYSTEM_PROMPT = 'Your role is to complete the command to continue the proof assitant.'

In [ ]:
from transformers import TextStreamer

In [ ]:
streamer = TextStreamer(tokenizer2)

In [ ]:
tokenizer2.tokenize(')\n')

[')Ċ']

In [ ]:
def generate(query):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': query}]
    tokens = tokenizer2.apply_chat_template(messages, return_tensors='pt',
                                           add_generation_prompt=True,
                                           return_dict=True)
    terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]

    with torch.inference_mode():
        out = model2.generate(**tokens.to("cuda"), max_new_tokens=100,
                             eos_token_id=terminators, top_p=0.5,
                             streamer=streamer)[0]

    return tokenizer2.decode(out)

prompt = """\
  |-------
{1}   FORALL (l1: list[T], x: T): length(leq_elements(l1, x)) <= length(l1)

Rule?"""
# _ = generate(prompt)

In [ ]:
end_tokens = tokenizer2([')\n', ')', '\n'], add_special_tokens=False).input_ids
end_tokens = sum(end_tokens, start=[1158])
end_tokens

[1158, 340, 8, 198]

In [ ]:
terminators = [tokenizer2.tokenize(text)[0] for text in [' "', ")\n"]]
terminators_ids = [tokenizer2.convert_tokens_to_ids(token) for token in terminators]

def generate2(query, max_new_tokens=20, stream=True):
    tokens = tokenizer2(query, return_tensors='pt').to("cuda")
    with torch.inference_mode():
        out = model2.generate(
            **tokens, max_new_tokens=max_new_tokens,
             eos_token_id=terminators_ids,
             streamer=streamer if stream else None,
             temperature=1,
             # top_p=0.5,
             top_k=1,
        )[0]

    return tokenizer2.decode(out)[len(query):]


In [ ]:
prompt = """\
{-1}  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(leq_elements(y, x)) <= length(y)
  |-------
{1}   FORALL (x: T): length(leq_elements(x!1, x)) <= length(x!1)

Rule?"""

_ = generate2(prompt)

<|begin_of_text|>{-1}  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(leq_elements(y, x)) <= length(y)
  |-------
{1}   FORALL (x: T): length(leq_elements(x!1, x)) <= length(x!1)

Rule? (SKEEP)



In [ ]:
prompt = """\
  |-------
{1}   FORALL (l1: list[T], x: T): length(leq_elements(l1, x)) <= length(l1)

Rule?"""

_ = generate2(prompt)

<|begin_of_text|>  |-------
{1}   FORALL (l1: list[T], x: T): length(leq_elements(l1, x)) <= length(l1)

Rule? (MEASURE-INDUCT+ "length(l1)" ("l1"))
  | ("l


In [ ]:
prompt = """\
[-1]  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(leq_elements(y, x)) <= length(y)
  |-------
{1}   length(CASES x!1
               OF null: null,
                  cons(x_1, r):
                    IF leq(x_1, x) THEN cons(x_1, leq_elements(r, x))
                    ELSE leq_elements(r, x)
                    ENDIF
               ENDCASES)
       <= length(x!1)

Rule?"""

_ = generate2(prompt)

<|begin_of_text|>[-1]  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(leq_elements(y, x)) <= length(y)
  |-------
{1}   length(CASES x!1
               OF null: null,
                  cons(x_1, r):
                    IF leq(x_1, x) THEN cons(x_1, leq_elements(r, x))
                    ELSE leq_elements(r, x)
                    ENDIF
               ENDCASES)
       <= length(x!1)

Rule? (LIFT-IF)



In [ ]:
prompt = """\
[-1]  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(g_elements(y, x)) <= length(y)
  |-------
{1}   length(CASES x!1
               OF null: null,
                  cons(x_1, r):
                    IF gt(x_1, x) THEN cons(x_1, g_elements(r, x))
                    ELSE g_elements(r, x)
                    ENDIF
               ENDCASES)
       <= length(x!1)

Rule?"""

_ = generate2(prompt)

<|begin_of_text|>[-1]  FORALL (y: list[T]):
        FORALL (x: T):
          length(y) < length(x!1) IMPLIES
           length(g_elements(y, x)) <= length(y)
  |-------
{1}   length(CASES x!1
               OF null: null,
                  cons(x_1, r):
                    IF gt(x_1, x) THEN cons(x_1, g_elements(r, x))
                    ELSE g_elements(r, x)
                    ENDIF
               ENDCASES)
       <= length(x!1)

Rule? (LIFT-IF)



# evaluating

In [ ]:
from tqdm.notebook import tqdm


tqdm.pandas()

In [ ]:
df_test = pd.read_csv('pvs-blocks-single-new-test.csv')
df_test

,samples,sequent,command,parameters
0,"[-1] geq(k, v`seq(x!1 - 1))\n[-2] leq(k, v`s...","[-1] geq(k, v`seq(x!1 - 1))\n[-2] leq(k, v`s...",BOTH-SIDES,"""-"" ""1"" -3"
1,{-1} k = 0\n[-2] k <= length[T](sorting_min(...,{-1} k = 0\n[-2] k <= length[T](sorting_min(...,REPLACES,-1
2,[-1] i < length(x!1)\n[-2] j < length(x!1)\n...,[-1] i < length(x!1)\n[-2] j < length(x!1)\n...,GRIND,NaN
3,|-------\n{1} FORALL ((l: list[T] | NOT nu...,|-------\n{1} FORALL ((l: list[T] | NOT nu...,SKOSIMP,*
4,|-------\n{1} t(y`1) > length(y`2) - 1\n{2...,|-------\n{1} t(y`1) > length(y`2) - 1\n{2...,ASSERT,NaN
...,...,...,...,...
1372,"|-------\n[1] insert?(y!1, x!2 ^ (1, lengt...","|-------\n[1] insert?(y!1, x!2 ^ (1, lengt...",DECOMPOSE-EQUALITY,NaN
1373,[-1] null?(x)\n[-2] n < length(x)\n |------...,[-1] null?(x)\n[-2] n < length(x)\n |------...,GRIND,NaN
1374,[-1] k!1 < 1 + x!2\n[-2] k!1 <= x!3\n[-3] k...,[-1] k!1 < 1 + x!2\n[-2] k!1 <= x!3\n[-3] k...,GRIND,NaN
1375,[-1] t(1 + k) < length(indx)\n[-2] j > indx`...,[-1] t(1 + k) < length(indx)\n[-2] j > indx`...,SPLIT,-3


In [ ]:
df_test.command.value_counts()

command
EXPAND                302
ASSERT                220
INST                  177
GRIND                 121
SKEEP                  94
PROP                   69
TYPEPRED               63
LEMMA                  55
FLATTEN                53
LIFT-IF                47
REPLACES               36
SPLIT                  30
CASE                   20
REPLACE                20
REWRITE                19
MEASURE-INDUCT         13
DECOMPOSE-EQUALITY      7
CASE-REPLACE            6
AUTO-REWRITE            5
BOTH-SIDES              4
INST-CP                 4
SKOSIMP                 3
USE                     3
SWAP-REL                2
BETA                    2
NAME-REPLACE            2
Name: count, dtype: int64

In [ ]:
df_test['output'] = df_test.sequent.progress_apply(generate2, max_new_tokens=6, stream=False)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [ ]:
def is_prediction_correct(sample):
    return re.split(r'\*|-|\?', sample.command)[0] in sample.output


df_test["is_correct"] = df_test.apply(is_prediction_correct, axis=1)
accuracty_top_1 = df_test["is_correct"].mean()
print(f"accuracty_top_1: {100 * accuracty_top_1:.2f}%")

accuracty_top_1: 53.30%


In [ ]:
EQUIV = {
    'SKOSIMP': ['SKEEP'],
    'REPLACES': ['REPLACE', 'ASSERT'],
    'REPLACE': ['REPLACES', 'ASSERT'],
    'REWRITE': ['LEMMA', 'USE'],
    'LEMMA': ['REWRITE', 'USE'],
    'SPLIT': ['PROP'],
    'FLATTEN': ['PROP'],
    'PROP': ['SPLIT', 'FLATTEN'],
    'INST-CP': ['INST', 'INST?']
}

def is_prediction_correct_soft(sample):
    return (re.split(r'\*|-|\?', sample.command)[0] in sample.output
            or any(equiv in sample.output for equiv in EQUIV.get(sample.command, [])))


df_test["is_correct2"] = df_test.apply(is_prediction_correct_soft, axis=1)
accuracty_top_1_soft =  df_test["is_correct2"].mean()
print(f"accuracty_top_1_soft: {100 * accuracty_top_1_soft:.2f}%")

accuracty_top_1_soft: 54.76%
